In [ ]:
import pandas as pd
import numpy as np
import os

path="."

In [ ]:
pip install serpapi

In [ ]:
import serpapi

client = serpapi.Client(api_key="YOUR_SERPAPI_KEY")
results = client.search({
  "engine": "google_maps_reviews",
  "hl": "en",
  "place_id": "ChIJL6u8cbXrwIcR-mZ3C6B147I"
})

In [ ]:
results ["reviews"]

In [ ]:
from serpapi import search
import pandas as pd

params = {
  "api_key": "YOUR_SERPAPI_KEY",
  "engine": "google_maps_reviews",
  "hl": "en",
  "place_id": "ChIJL6u8cbXrwIcR-mZ3C6B147I"
}
res = []
for x in range(20):
  print(x)
  results = search(params)
  if "error" in results:
    print(results["error"])
    break
  next_page_token  = results["serpapi_pagination"]["next_page_token"]
  res.append(results)
  params["next_page_token"] = next_page_token
reviews = []
for x in res:
  reviews.append(pd.DataFrame(x["reviews"]))
reviews = pd.concat(reviews)

In [ ]:
reviews = pd.read_csv ('reviews.csv')

Analysis

In [ ]:
reviews.columns

In [ ]:
reviews.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style
sns.set_theme(style="whitegrid")

# Create the histogram
plt.figure(figsize=(10, 6))
sns.histplot(reviews['rating'], bins=5, kde=False, color='skyblue')

# Add labels and title
plt.title('Distribution of Ratings', fontsize=15)
plt.xlabel('Rating', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks([1, 2, 3, 4, 5])

# Show the plot
plt.show()

In [ ]:
import nltk
nltk.download('stopwords', quiet=True) # Download stopwords if not already present
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.decomposition import LatentDirichletAllocation as LDA

# --- Step 1: Define the Corpus ---
# The text data we want to analyze is in the 'snippet' column of the 'reviews' DataFrame.
corpus = reviews['snippet'].astype(str) # Ensure snippets are strings

# --- Step 2: Text Vectorization (CountVectorizer) ---
# Convert a collection of text documents to a matrix of token counts.
# We'll remove common English stopwords and some additional words that might not be discriminative.
# We set a minimum document frequency to ignore words that appear too rarely (e.g., in less than 2 documents).
custom_stopwords = ['restaurant', 'buffalowildwings', 'bww', 'place', 'like', 'get', 'got', 'go', 'us', 'staff', 'manager', 'just', 'really', 'much']

count_vect = CountVectorizer(
    stop_words=stopwords.words('english') + custom_stopwords,
    lowercase=True,
    min_df=2 # Ignore terms that appear in less than 2 documents
)
x_counts = count_vect.fit_transform(corpus)
feature_names = count_vect.get_feature_names_out()
print(f"Number of unique words (features) after vectorization: {len(feature_names)}")

# --- Step 3: TF-IDF Transformation ---
# Transform count matrix to a normalized tf-idf representation.
# TF-IDF stands for Term Frequency-Inverse Document Frequency.
# It weighs words based on how frequently they appear in a document relative to how frequently they appear across all documents.
tfidf_transformer = TfidfTransformer()
x_tfidf = tfidf_transformer.fit_transform(x_counts)
print(f"TF-IDF matrix shape: {x_tfidf.shape}")

# --- Step 4: Apply Latent Dirichlet Allocation (LDA) ---
# LDA is a generative probabilistic model for collections of discrete data.
# It's a topic model that is used to discover the abstract "topics" that occur in a collection of documents.

# Define the number of topics (categories) we want to find.
# You can experiment with different numbers of topics (e.g., 3, 5, 7, 10).
num_topics = 5

lda = LDA(n_components=num_topics, random_state=42) # Set random_state for reproducibility
lda.fit(x_tfidf)

# --- Step 5: Display the Top Words for Each Topic ---
# For each topic, we'll extract the words that contribute most to it.

def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_): # Corrected from model.components__
        print(f"\nTopic {topic_idx + 1}:")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))

# Define how many top words to show for each topic
no_top_words = 10
display_topics(lda, feature_names, no_top_words)

# NER

In [ ]:
import pandas as pd
import spacy

# Install the 'en_core_web_lg' model if not already present
!python -m spacy download en_core_web_lg --quiet

nlp = spacy.load("en_core_web_lg")
pd.set_option("display.max_rows", 200)
from spacy import displacy

# Ensure 'reviews' DataFrame is loaded if not already defined for this part of the analysis
try:
    reviews
except NameError:
    # Assuming reviews.csv is located at 'reviews.csv'
    reviews = pd.read_csv ('reviews.csv')


In [ ]:
# This cell is no longer needed as 'en_core_web_lg' is loaded in the previous cell.
# Keeping the previous, larger model for consistency.
# nlp = spacy.load("en_core_web_sm")
# loaded spacy model first

In [ ]:
# This cell is no longer needed as the 'reviews' DataFrame is consistently used.
# df = pd.read_csv('reviews.csv')
# copied reviews

In [ ]:
def get_entities(text):
    if not isinstance(text, str):
        return []
    doc = nlp(text)
    # Returns a list of tuples: (entity_text, entity_label)
    return [(ent.text, ent.label_) for ent in doc.ents]


In [ ]:
# This cell will be replaced by the application of the get_ents function (from 2Q3rq7wGAyNm) in cell jGgNBmAoBWq_
# reviews['get_entities'] = reviews['snippet'].apply(get_entities)

In [ ]:
print(df[['value', 'type']].head(20))

In [ ]:
df.to_csv('reviews_with_entities.csv', index=False)

In [ ]:
import spacy

# 1. Load the model (Run !python -m spacy download en_core_web_lg if not already done)
# The model is now loaded in cell x01L_gfJY7EO
# nlp = spacy.load("en_core_web_lg")

# 2. Extract entities (Replace 'review_column' with your actual column name)
def get_ents(column):
    # Ensure the 'reviews' DataFrame is available
    global reviews
    try:
        reviews
    except NameError:
        reviews = pd.read_csv('reviews.csv')

    # Use nlp.pipe for efficient processing of a Series
    return [[(ent.text, ent.label_) for ent in doc.ents]
            for doc in nlp.pipe(column.astype(str))]


### Most Frequent Entities per Type

Let's find the top 10 most frequently mentioned specific entities for a few interesting types, like 'PERSON', 'GPE' (geopolitical entities), and 'ORG' (organizations).

In [ ]:
from collections import Counter

def get_top_entities_by_type(entity_type, num_top=10):
    # Filter entities_df for the specific type, convert values to lowercase for case-insensitive counting
    filtered_entities = entities_df[entities_df['type'] == entity_type]['value'].str.lower()

    # Count the occurrences of each entity value
    entity_counts = Counter(filtered_entities)

    print(f"\nTop {num_top} '{entity_type}' entities:")
    for entity, count in entity_counts.most_common(num_top):
        print(f"- {entity}: {count}")

# Example usage:
get_top_entities_by_type('PERSON')
get_top_entities_by_type('GPE')
get_top_entities_by_type('ORG')
get_top_entities_by_type('MONEY')
get_top_entities_by_type('PRODUCT')

### Average Rating by Entity

Let's calculate the average rating for reviews where specific entities are mentioned. This can give insights into sentiment associated with particular entities.

In [ ]:
# Calculate the average rating for each unique entity_value
average_rating_by_entity = entities_with_review_context_df.groupby('entity_value')['rating'].mean().sort_values(ascending=False)

print("Top 10 Entities by Average Rating (highest):")
display(average_rating_by_entity.head(10))

print("\nTop 10 Entities by Average Rating (lowest):")
display(average_rating_by_entity.tail(10))

In [ ]:
# Apply and view
# Ensure the 'reviews' DataFrame is available, as sentiment analysis might have reloaded it.
# If this cell is run independently, ensure 'reviews' is loaded. (Handled by the sentiment cell)

reviews['entities'] = get_ents(reviews['snippet'])
print(reviews[['snippet', 'entities']].head())

In [ ]:
all_entities = []
for entity_list in reviews['entities']:
    all_entities.extend(entity_list)

entities_df = pd.DataFrame(all_entities, columns=['value', 'type'])
display(entities_df.head())

In [ ]:
import pandas as pd
import numpy as np # Import numpy

# Initialize an empty list to store the new dataset records
entity_review_data = []

# Iterate through each row of the reviews DataFrame
for index, row in reviews.iterrows():
    review_id = row['review_id']
    rating = row['rating']
    # Ensure sentiment columns exist. If not, default to NaN or re-calculate
    sentiment_compound = row['sentiment_compound'] if 'sentiment_compound' in row else np.nan
    date = row['date']
    iso_date = row['iso_date']
    snippet = row['snippet']

    # Get the list of entities for the current review
    entities_in_review = row['entities']

    # For each entity, create a new record with review details
    for entity_value, entity_type in entities_in_review:
        entity_review_data.append({
            'entity_value': entity_value,
            'entity_type': entity_type,
            'review_id': review_id,
            'rating': rating,
            'sentiment_compound': sentiment_compound,
            'date': date,
            'iso_date': iso_date,
            'snippet': snippet
        })

# Create a new DataFrame from the collected data
entities_with_review_context_df = pd.DataFrame(entity_review_data)

# Display the first few rows of the new DataFrame
display(entities_with_review_context_df.head())

In [ ]:
from spacy import displacy

# Take the first review's snippet for visualization
sample_review_snippet = reviews['snippet'].iloc[0]
doc = nlp(sample_review_snippet)

# Visualize the entities
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
# I am correcting 'Dakota' from a PERSON to a GPE (geopolitical entity/location)
entities_df.loc[entities_df['value'].str.contains('Dakota', case=False), 'type'] = 'GPE'

# I am correcting 'Maryam' from a GPE to a PERSON
entities_df.loc[entities_df['value'].str.contains('Maryam', case=False), 'type'] = 'PERSON'

# Display the first few rows to show the updated entity types
print("Updated entities_df:")
display(entities_df.head(10))

Corrections

In [ ]:
# I am applying the same corrections to the entities_with_review_context_df DataFrame
# Correcting 'Dakota' from a PERSON to a GPE (geopolitical entity/location)
entities_with_review_context_df.loc[entities_with_review_context_df['entity_value'].str.contains('Dakota', case=False), 'entity_type'] = 'GPE'

# Correcting 'Maryam' from a GPE to a PERSON
entities_with_review_context_df.loc[entities_with_review_context_df['entity_value'].str.contains('Maryam', case=False), 'entity_type'] = 'PERSON'

# Display the first few rows to show the updated entity types with review context
print("\nUpdated entities_with_review_context_df:")
display(entities_with_review_context_df.head(10))

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import spacy
from collections import Counter

# --- 0. Ensure NLTK and SpaCy resources are ready ---
nltk.download('vader_lexicon', quiet=True)
# Assuming spacy model 'en_core_web_lg' is already loaded by cell x01L_gfJY7EO
# and get_ents function is defined in cell 2Q3rq7wGAyNm

# --- 1. Load Reviews DataFrame ---
reviews = pd.read_csv('reviews.csv')

# --- 2. Perform Sentiment Analysis (re-running logic from 0bezTFDoiiSP) ---
sid = SentimentIntensityAnalyzer()
def sentiment(review):
  if(not isinstance(review, str)):
    return [np.nan,np.nan,np.nan,np.nan]
  sentiment_score = sid.polarity_scores(review)
  return [sentiment_score['neg'],sentiment_score['neu'],sentiment_score['pos'],sentiment_score['compound']]
reviews[['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound']] = reviews['snippet'].apply(sentiment).apply(pd.Series)
print("Reviews DataFrame updated with sentiment scores.")

# --- 3. Extract Entities (re-running logic from jGgNBmAoBWq_) ---
# Ensure nlp and get_ents are defined and available
try:
    nlp
except NameError:
    print("Loading spacy model in orchestration cell...")
    !python -m spacy download en_core_web_lg --quiet
    nlp = spacy.load("en_core_web_lg")

def get_ents(column):
    return [[(ent.text, ent.label_) for ent in doc.ents]
            for doc in nlp.pipe(column.astype(str))]

reviews['entities'] = get_ents(reviews['snippet'])
print("Reviews DataFrame updated with extracted entities.")

# --- 4. Create entities_df (re-running logic from 7c151803) ---
all_entities = []
for entity_list in reviews['entities']:
    all_entities.extend(entity_list)
entities_df = pd.DataFrame(all_entities, columns=['value', 'type'])
print("entities_df created.")

# --- 5. Correct entity types in entities_df (re-running logic from ec004dfb) ---
entities_df.loc[entities_df['value'].str.contains('Dakota', case=False), 'type'] = 'GPE'
entities_df.loc[entities_df['value'].str.contains('Maryam', case=False), 'type'] = 'PERSON'
print("entities_df corrected for 'Dakota' and 'Maryam'.")

# --- 6. Create entities_with_review_context_df (re-running logic from 38db1e73) ---
entity_review_data = []
for index, row in reviews.iterrows():
    review_id = row['review_id']
    rating = row['rating']
    sentiment_compound = row['sentiment_compound'] # Should now always be present
    date = row['date']
    iso_date = row['iso_date']
    snippet = row['snippet']

    entities_in_review = row['entities']

    for entity_value, entity_type in entities_in_review:
        entity_review_data.append({
            'entity_value': entity_value,
            'entity_type': entity_type,
            'review_id': review_id,
            'rating': rating,
            'sentiment_compound': sentiment_compound,
            'date': date,
            'iso_date': iso_date,
            'snippet': snippet
        })
entities_with_review_context_df = pd.DataFrame(entity_review_data)
print("entities_with_review_context_df created.")

# --- 7. Correct entity types in entities_with_review_context_df (re-running logic from b09fd2b5) ---
entities_with_review_context_df.loc[entities_with_review_context_df['entity_value'].str.contains('Dakota', case=False), 'entity_type'] = 'GPE'
entities_with_review_context_df.loc[entities_with_review_context_df['entity_value'].str.contains('Maryam', case=False), 'entity_type'] = 'PERSON'
print("entities_with_review_context_df corrected for 'Dakota' and 'Maryam'.")

print("\nAll dataframes prepared and ready for entity sentiment analysis!")

VISUALIZATION

In [ ]:
# Get the count of each entity type
entity_type_counts = entities_df['type'].value_counts()

print("Distribution of Entity Types:")
display(entity_type_counts)

# Visualize the distribution of entity types
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.barplot(x=entity_type_counts.index, y=entity_type_counts.values, palette='viridis')
plt.title('Count of Each Entity Type')
plt.xlabel('Entity Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

def plot_top_entities(entity_type, df, num_top=10):
    # Filter entities_df for the specific type, convert values to lowercase for case-insensitive counting
    filtered_entities = df[df['type'] == entity_type]['value'].str.lower()

    # Count the occurrences of each entity value
    entity_counts = Counter(filtered_entities)

    # Get the top N entities
    top_entities_data = entity_counts.most_common(num_top)

    if not top_entities_data:
        print(f"No '{entity_type}' entities found to plot.")
        return

    # Convert to DataFrame for easier plotting with seaborn
    plot_df = pd.DataFrame(top_entities_data, columns=['Entity', 'Count'])

    plt.figure(figsize=(10, 6))
    sns.barplot(x='Count', y='Entity', data=plot_df, palette='viridis')
    plt.title(f'Top {num_top} Most Frequent {entity_type} Entities', fontsize=15)
    plt.xlabel('Count', fontsize=12)
    plt.ylabel(entity_type, fontsize=12)
    plt.tight_layout()
    plt.show()

# Generate plots for specific entity types
plot_top_entities('PERSON', entities_df)


Findings


Based on my descriptive analysis of the reviews, I've observed a few key points:

*   **Rating Distribution:** The distribution of ratings indicates that there are more 1-star and 5-star reviews than 2, 3, or 4-star reviews. This suggests customers either being very satisfied or very dissatisfied.
*   **Review Length and Likes:** There doesn't appear to be a strong linear relationship between review length and the number of likes received.
*   **Entity Distribution:** In terms of named entities, 'CARDINAL' (numbers), 'TIME', and 'DATE' are the most frequently extracted types, followed by 'ORG' (organizations).

In [ ]:
import pandas as pd

# Save the DataFrame to a CSV file in Google Drive
output_path = 'name_entities.csv'
entities_with_review_context_df.to_csv(output_path, index=False)

print(f"DataFrame successfully exported to {output_path}")

SENTIMENTS ANALYSIS

In [ ]:
!pip install nltk textblob

In [ ]:
import nltk
import numpy as np
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd

# Load the reviews DataFrame to ensure it's available
# Assuming reviews.csv is located at 'reviews.csv'
reviews = pd.read_csv('reviews.csv')

nltk.download('vader_lexicon')
sid = SentimentIntensityAnalyzer()

def sentiment(review):

  if(type(review) != str):
    return [np.nan,np.nan,np.nan,np.nan]

  sentiment_score = sid.polarity_scores(review)
  return [sentiment_score['neg'],sentiment_score['neu'],sentiment_score['pos'],sentiment_score['compound']]

reviews[['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound']] = reviews['snippet'].apply(sentiment).apply(pd.Series)

In [ ]:
s = sentiment("iheard he was happy,butiam not happy")
print('negative:',s[0])
print("neutral:",s[1])
print("positive:",s[2])
print("compound:",s[3])

In [ ]:
#  10 phrases for sentiment reveiew

In [ ]:
sentiment_phrases = [
    "This restaurant is absolutely amazing! The food was delicious and the service was impeccable.",
    "The movie was okay, but nothing special. I wouldn't recommend it to everyone.",
    "I had a terrible experience. The product broke after one use and the customer service was unhelpful.",
    "What a pleasant surprise! I really enjoyed my time there and will definitely be back.",
    "It was neither good nor bad. Just average.",
    "I'm so disappointed with this purchase. It's a complete waste of money.",
    "The concert was fantastic! The band played all my favorite songs.",
    "The wait time was a bit long, but the staff was friendly.",
    "Absolutely horrible! I'm never going there again.",
    "Pretty good, exceeded my expectations for the price."
]

# You can now iterate through these phrases and apply your sentiment function
for i, phrase in enumerate(sentiment_phrases):
    s = sentiment(phrase)
    print(f"Phrase {i+1}: '{phrase}'")
    print(f"  Negative: {s[0]}, Neutral: {s[1]}, Positive: {s[2]}, Compound: {s[3]}\n")

In [ ]:
reviews[['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound']].describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ax = sns.histplot(reviews['sentiment_neg'])
ax.set_title('Negative Sentiment')
plt.show()

ax = sns.histplot(reviews['sentiment_pos'])
ax.set_title('Positive Sentiment')
plt.show()

ax = sns.histplot(reviews['sentiment_neu'])
ax.set_title('Neutral Sentiment')
plt.show()

#SECOND METHOD

In [ ]:
from transformers import pipeline
import numpy as np

# Load a pre-trained sentiment analysis pipeline
sentiment_pipeline = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')

def bert_sentiment(review):
    if not isinstance(review, str):
        return [np.nan, np.nan, np.nan] # Return NaN for non-string input

    result = sentiment_pipeline(review)[0]
    label = result['label']
    score = result['score']
    return [label, score]

### Analyzing Sentiment vs. Review Ratings

To understand if the sentiment metrics (specifically the compound score from VADER) align with the numerical review ratings, we will perform the following:

1.  **Calculate Correlation**: Compute the Pearson correlation coefficient between the `rating` and `sentiment_compound` columns.
2.  **Visualize Distribution**: Create a box plot to visualize the distribution of `sentiment_compound` scores for each `rating` level (1-5 stars). This will help us see if higher ratings generally correspond to more positive compound sentiment scores.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate the correlation between 'rating' and 'sentiment_compound'
correlation = reviews['rating'].corr(reviews['sentiment_compound'])
print(f"Correlation between Rating and Compound Sentiment Score: {correlation:.2f}")

# Create a box plot to visualize sentiment distribution per rating
plt.figure(figsize=(10, 6))
sns.boxplot(x='rating', y='sentiment_compound', data=reviews, palette='viridis')
plt.title('Compound Sentiment Score Distribution by Review Rating', fontsize=15)
plt.xlabel('Review Rating', fontsize=12)
plt.ylabel('Compound Sentiment Score', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Findings: Sentiment vs. Review Ratings

Based on the analysis:

*   **Correlation**: The calculated Pearson correlation coefficient between `rating` and `sentiment_compound` is **0.63**. A positive correlation would suggest that as review ratings increase, the sentiment compound score also tends to increase, indicating alignment.

*   **Box Plot Visualization**: The box plot shows the distribution of compound sentiment scores for each star rating. Generally, we would expect to see:
    *   Lower ratings (e.g., 1-star) to have predominantly negative compound scores.
    *   Higher ratings (e.g., 5-star) to have predominantly positive compound scores.
    *   Mid-range ratings (e.g., 2, 3, 4-star) might have a wider spread of sentiment scores, or scores closer to neutral, depending on the nuances of the reviews.

This visualization helps confirm if the lexicon-based sentiment analysis is capturing the positive/negative tone of the reviews in a way that matches the explicit star ratings given by users. If the boxes for 1-star reviews are largely below 0 (negative) and 5-star reviews are largely above 0 (positive), it suggests a good alignment.

# TOPIC MODELING

## Topic Modeling and Visualization

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.decomposition import LatentDirichletAllocation as LDA
from nltk.corpus import stopwords
import nltk, pandas as pd, numpy as np

nltk.download('stopwords')

extra_stopwords = ['wings','buffalo','wild','bww','restaurant','place','food','order','get','got','like']

def run_lda(corpus, theme_nums=[3,4,5,6], top_words=8):
    corpus = corpus.dropna().astype(str)

    count_vect = CountVectorizer(
        stop_words=stopwords.words('english') + extra_stopwords,
        lowercase=True,
        min_df=2
    )

    x_counts = count_vect.fit_transform(corpus)
    feature_names = count_vect.get_feature_names_out()

    tfidf = TfidfTransformer()
    x_tfidf = tfidf.fit_transform(x_counts)

    for n in theme_nums:
        print(f"\n===== {n} THEMES =====")
        lda = LDA(n_components=n, random_state=42)
        lda.fit(x_tfidf)

        for i, topic in enumerate(lda.components_):
            words = [feature_names[j] for j in topic.argsort()[:-top_words-1:-1]]
            print(f"Theme {i+1}: {', '.join(words)}")

    return lda, x_tfidf, feature_names

In [ ]:
lda_all, x_all, words_all = run_lda(reviews['snippet'], theme_nums=[3,4,5,6])

In [ ]:
dimension = 3
number_of_words_per_theme = 6

#identifying themes
lda = LDA(n_components = dimension)
lda_array = lda.fit_transform(x_tfidf)

components = [lda.components_[i] for i in range(len(lda.components_))]
important_words = [sorted(feature_names, key = lambda x: components[j][np.where(feature_names == x)], reverse = True)[:number_of_words_per_theme] for j in range(len(components))]
important_words


In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import numpy as np
import pandas as pd

# Ensure NLTK vader_lexicon is downloaded if not already
nltk.download('vader_lexicon', quiet=True)

# Initialize SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer()

# Define the sentiment function
def sentiment(review):
  if not isinstance(review, str):
    return [np.nan, np.nan, np.nan, np.nan]
  sentiment_score = sid.polarity_scores(review)
  return [sentiment_score['neg'], sentiment_score['neu'], sentiment_score['pos'], sentiment_score['compound']]

# Calculate sentiment scores and add them to the reviews DataFrame
reviews[['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound']] = reviews['snippet'].apply(sentiment).apply(pd.Series)

cutoff = int(len(reviews) * 0.30)

negative_reviews = reviews.sort_values(
    by='sentiment_neg',
    ascending=False
).head(cutoff)

lda_neg, x_neg, words_neg = run_lda(negative_reviews['snippet'], theme_nums=[3,4,5,6])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Get the topic distribution for each document
# lda_array was already computed in the previous cell
topic_distributions = lda.transform(x_tfidf)

# Assign each review to its dominant topic
dominant_topic = np.argmax(topic_distributions, axis=1)

# Map the numerical topics to descriptive names based on our interpretation
topic_names = {
    0: 'Food Quality/Taste',
    1: 'Service Experience',
    2: 'Atmosphere/Ambiance',
    3: 'Price/Value',
    4: 'Order/Wait Times'
}

# Create a DataFrame for plotting
topic_counts = pd.Series(dominant_topic).map(topic_names).value_counts().reset_index()
topic_counts.columns = ['Topic', 'Number of Reviews']

# Sort topics by count for better visualization
topic_counts = topic_counts.sort_values(by='Number of Reviews', ascending=False)

# Create the bar chart
plt.figure(figsize=(12, 7))
# Corrected line to address FutureWarning
sns.barplot(x='Number of Reviews', y='Topic', hue='Topic', data=topic_counts, palette='viridis', legend=False)

# Add labels and title
plt.title('Distribution of Reviews Across Identified Topics', fontsize=16)
plt.xlabel('Number of Reviews', fontsize=12)
plt.ylabel('Topic', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

For all reviews, I tested 3, 4, 5, and 6 themes using CountVectorizer, TF-IDF transformation, and LDA topic modeling. English stop words and custom stop words such as “wings,” “buffalo,” “wild,” and “food” were removed to make the themes clearer. The final themes show the main customer discussion areas, including service, wait time, food quality, pricing/value, and atmosphere.

For the top 30% most negative reviews, the same method was repeated. The negative review themes showed stronger focus on slow service, incorrect orders, cold or dry food, wait time, and poor customer experience. These themes identify the main business problems Buffalo Wild Wings should address.

TEXT SUMMARIZATION

In [ ]:
!pip install pytextrank
import spacy
import pytextrank



In [ ]:
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("textrank")

In [ ]:
def get_extractive_summary(text, limit_sentences=1):
    if not isinstance(text, str) or not text.strip():
        return ""
    doc = nlp(text)
    # Ensure the TextRank pipeline has been added to the nlp object
    if 'textrank' not in nlp.pipe_names:
        nlp.add_pipe("textrank")
    summary_sentences = doc._.textrank.summary(limit_sentences=limit_sentences)
    return ' '.join([s.text for s in summary_sentences])

# Apply the extractive summarization to the 'snippet' column
reviews['extractive_summary'] = reviews['snippet'].apply(get_extractive_summary)

print("Reviews DataFrame with new 'extractive_summary' column:")
display(reviews[['snippet', 'extractive_summary']].head())

In [ ]:
reviews['length'] = reviews['snippet'].astype(str).apply(lambda x: len(x))
review_text = reviews.sort_values(by='length', ascending=False)['snippet'].values[2]
review_text

#Extractive summarization using PageRank

In [ ]:
doc = nlp(review_text)
extractive_summary = doc._.textrank.summary(limit_sentences=1, limit_phrases=5)
line = 0
for sentence in extractive_summary:
  display(sentence)
  line += 1

In [ ]:
for phrase in doc._.phrases[0:10]:
    print(phrase.text, phrase.rank, phrase.count)
    print(phrase.chunks)

#Abstractive summarization using Pegasus transformer

In [ ]:
from transformers import pipeline
from transformers import PegasusForConditionalGeneration, PegasusTokenizer

model_name = "google/pegasus-xsum"
#if you connect Colab to a compute engine with GPU this code will show the environment's parameters


In [ ]:
!nvidia-smi

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
pegasus_model = PegasusForConditionalGeneration.from_pretrained(model_name).to(device)
pegasus_tokenizer = PegasusTokenizer.from_pretrained(model_name)

def pegasus_summarize(text, max_len):
  tokens = pegasus_tokenizer(text, truncation=True, padding="longest", return_tensors="pt").to(device)
  encoded_summary = pegasus_model.generate(**tokens, max_length=max_len)
  decoded_summary = pegasus_tokenizer.decode(encoded_summary[0], skip_special_tokens=True)
  return decoded_summary

In [ ]:
max_len = 64
abstractive_summary = pegasus_summarize(review_text, max_len)
abstractive_summary

In [ ]:
import pandas as pd
import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer
import spacy

# Install pytextrank if not already present
!pip install pytextrank
import pytextrank

# Ensure the reviews DataFrame is loaded
try:
    reviews
except NameError:
    reviews = pd.read_csv('reviews.csv')

# --- SpaCy Model Setup for Extractive Summarization ---
# It's good practice to ensure the model is downloaded if not already
!python -m spacy download en_core_web_lg # Uncomment and run if 'en_core_web_lg' is not installed
nlp = spacy.load("en_core_web_lg")
if "textrank" not in nlp.pipe_names:
    nlp.add_pipe("textrank")
# --------------------------------------------------------

# --- Pegasus Model Setup (copied from previous cell for self-containment) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device for summarization: {device}")

model_name = "google/pegasus-xsum"
pegasus_model = PegasusForConditionalGeneration.from_pretrained(model_name).to(device)
pegasus_tokenizer = PegasusTokenizer.from_pretrained(model_name)

def pegasus_summarize(text, max_len):
  tokens = pegasus_tokenizer(text, truncation=True, padding="longest", return_tensors="pt").to(device)
  encoded_summary = pegasus_model.generate(**tokens, max_length=max_len)
  decoded_summary = pegasus_tokenizer.decode(encoded_summary[0], skip_special_tokens=True)
  return decoded_summary
# ---------------------------------------------------------------------------

# Select the first 5 reviews for summarization
sample_reviews = reviews['snippet'].head(5)

print("--- Summarizing the first 5 reviews ---")

# Define new parameters
extractive_limit_sentences = 2  # Changed from 1
abstractive_max_len = 70      # Changed from 64

for i, review_text_to_summarize in enumerate(sample_reviews):
    print(f"\nReview {i+1}:")
    print(f"Original Review (first 200 chars): {review_text_to_summarize[:200]}...")

    # Extractive Summarization
    # Ensure the text is processed by spacy's nlp model with textrank pipeline
    doc_extractive = nlp(review_text_to_summarize)
    extractive_summary_sentences = doc_extractive._.textrank.summary(limit_sentences=extractive_limit_sentences) # Use new parameter
    extractive_summary = ' '.join([s.text for s in extractive_summary_sentences])
    print(f"Extractive Summary: {extractive_summary}")

    # Abstractive Summarization
    abstractive_summary = pegasus_summarize(review_text_to_summarize, max_len=abstractive_max_len) # Use new parameter
    print(f"Abstractive Summary: {abstractive_summary}")
    print("---------------------------------------------------")

In [ ]:
import pandas as pd
import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer

# Ensure the reviews DataFrame is loaded
try:
    reviews
except NameError:
    reviews = pd.read_csv('reviews.csv')

# --- Pegasus Model Setup (copied from previous cell for self-containment) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device for summarization: {device}")

model_name = "google/pegasus-xsum"
pegasus_model = PegasusForConditionalGeneration.from_pretrained(model_name).to(device)
pegasus_tokenizer = PegasusTokenizer.from_pretrained(model_name)

def pegasus_summarize(text, max_len):
  tokens = pegasus_tokenizer(text, truncation=True, padding="longest", return_tensors="pt").to(device)
  encoded_summary = pegasus_model.generate(**tokens, max_length=max_len)
  decoded_summary = pegasus_tokenizer.decode(encoded_summary[0], skip_special_tokens=True)
  return decoded_summary
# ---------------------------------------------------------------------------

max_len = 64 # Ensure max_len is defined for consistency

print("Generating abstractive summaries for all reviews. This might take a moment...")
reviews['abstractive_summary_all'] = reviews['snippet'].apply(lambda x: pegasus_summarize(x, max_len=max_len))
print("Abstractive summaries for all reviews generated.")

print("Displaying the first 5 reviews with their new abstractive summaries:")
display(reviews[['snippet', 'abstractive_summary_all']].head())

5 TABULATION REVIEWS OF BOTH ETRACTITVE AND ABSTRACTIVE SUMMARIES

In [ ]:
summaries_data = []

for i, review_text_to_summarize in enumerate(sample_reviews):
    doc_extractive = nlp(review_text_to_summarize)
    extractive_summary_sentences = doc_extractive._.textrank.summary(limit_sentences=1)
    extractive_summary = ' '.join([s.text for s in extractive_summary_sentences])

    abstractive_summary = pegasus_summarize(review_text_to_summarize, max_len=64)

    summaries_data.append({
        'Review Number': i + 1,
        'Original Review (Snippet)': review_text_to_summarize[:150] + '...',
        'Extractive Summary': extractive_summary,
        'Abstractive Summary': abstractive_summary
    })

summaries_df = pd.DataFrame(summaries_data)
display(summaries_df)

In [ ]:
output_excel_path = 'summaries.xlsx'
summaries_df.to_excel(output_excel_path, index=False)
print(f"DataFrame successfully exported to {output_excel_path}")

10 LONGEST REVIEWS WTH THE HIGHEST REVIEWS AND 1O LONGEST REVIEW WITH THE LOWEST RATINGS

In [ ]:
import nltk # Ensure nltk is imported for sentence tokenization if not already
import pandas as pd # Ensure pandas is imported
import os

nltk.download('punkt_tab', quiet=True) # Download the missing NLTK resource

def remove_consecutive_duplicate_words(text):
    if not isinstance(text, str):
        return text
    words = text.split()
    if not words:
        return ""
    cleaned_words = [words[0]]
    for i in range(1, len(words)):
        if words[i].lower() != cleaned_words[-1].lower(): # Compare with the last added word
            cleaned_words.append(words[i])
    return " ".join(cleaned_words)

def remove_duplicate_sentences(text):
    if not isinstance(text, str):
        return text
    sentences = nltk.sent_tokenize(text)
    unique_sentences = []
    seen_sentences = set()
    for sentence in sentences:
        if sentence.strip().lower() not in seen_sentences:
            unique_sentences.append(sentence.strip())
            seen_sentences.add(sentence.strip().lower())
    return ' '.join(unique_sentences)

# Define output_directory
output_directory = 'reviews.csv'

summary_records_high_low = []

# Summarize Highest Rated Reviews
for i, row in highest_rated_longest_reviews.iterrows(): # Corrected variable name
    review_text = row['snippet']
    if isinstance(review_text, str):
        # Extractive Summary
        doc_extractive = nlp(review_text)
        extractive_summary_sentences = []
        for sentence in doc_extractive._.textrank.summary(limit_sentences=2):
            extractive_summary_sentences.append(str(sentence))
        extractive_summary = " ".join(extractive_summary_sentences)
        extractive_summary = remove_duplicate_sentences(extractive_summary)
        extractive_summary = remove_consecutive_duplicate_words(extractive_summary)

        # Abstractive Summary
        max_len = 50 # Adjust max length as needed
        abstractive_summary = pegasus_summarize(review_text, max_len)
        abstractive_summary = remove_duplicate_sentences(abstractive_summary)
        abstractive_summary = remove_consecutive_duplicate_words(abstractive_summary)

        summary_records_high_low.append({
            'Review_Type': 'Highest Rated',
            'Original_Snippet': review_text,
            'Rating': row['rating'],
            'Length': row['length'],
            'Extractive_Summary': extractive_summary,
            'Abstractive_Summary': abstractive_summary
        })

# Summarize Lowest Rated Reviews
for i, row in lowest_rated_longest_reviews.iterrows(): # Corrected variable name
    review_text = row['snippet']
    if isinstance(review_text, str):
        # Extractive Summary
        doc_extractive = nlp(review_text)
        extractive_summary_sentences = []
        for sentence in doc_extractive._.textrank.summary(limit_sentences=2):
            extractive_summary_sentences.append(str(sentence))
        extractive_summary = " ".join(extractive_summary_sentences)
        extractive_summary = remove_duplicate_sentences(extractive_summary)
        extractive_summary = remove_consecutive_duplicate_words(extractive_summary)

        # Abstractive Summary
        max_len = 50 # Adjust max length as needed
        abstractive_summary = pegasus_summarize(review_text, max_len)
        abstractive_summary = remove_duplicate_sentences(abstractive_summary)
        abstractive_summary = remove_consecutive_duplicate_words(abstractive_summary)

        summary_records_high_low.append({
            'Review_Type': 'Lowest Rated',
            'Original_Snippet': review_text,
            'Rating': row['rating'],
            'Length': row['length'],
            'Extractive_Summary': extractive_summary,
            'Abstractive_Summary': abstractive_summary
        })

summary_high_low_df = pd.DataFrame(summary_records_high_low)
display(summary_high_low_df)

# Optionally, save this to an Excel file as well
output_file_high_low = os.path.join(output_directory, 'summarized_high_low_reviews.xlsx')
summary_high_low_df.to_excel(output_file_high_low, index=False)
print(f"Summaries of highest and lowest rated reviews exported to {output_file_high_low}")

In [ ]:
# Ensure 'review_length' is available
if 'review_length' not in reviews.columns:
    reviews['review_length'] = reviews['snippet'].astype(str).apply(len)

# 10 longest reviews with the highest ratings (e.g., 5-star)
highest_rated_longest_reviews = reviews[reviews['rating'] == 5].sort_values(by='review_length', ascending=False).head(10)

print("\n--- 10 Longest Reviews with 5-Star Ratings ---")
display(highest_rated_longest_reviews[['rating', 'review_length', 'snippet']])

# 10 longest reviews with the lowest ratings (e.g., 1-star)
lowest_rated_longest_reviews = reviews[reviews['rating'] == 1].sort_values(by='review_length', ascending=False).head(10)

print("\n--- 10 Longest Reviews with 1-Star Ratings ---")
display(lowest_rated_longest_reviews[['rating', 'review_length', 'snippet']])

In [ ]:
import collections

def remove_duplicate_words(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    words = text.split()
    # Use OrderedDict to preserve order while keeping only unique elements
    return ' '.join(collections.OrderedDict.fromkeys(words))

## Experience/Challenges Setting Up and Running Transformer Models

Setting up and running transformer models, especially for tasks like abstractive summarization, can present several challenges and learning opportunities:

1.  **Computational Resources:** Transformer models, such as Pegasus, are often very large and computationally intensive. Running them efficiently typically requires a GPU. In this notebook, we observed that a GPU was not available (`!nvidia-smi` failed), leading to the models running on the CPU. While functional, this can significantly increase processing time, especially for larger datasets or more complex operations.

2.  **Library Management and Dependencies:** Ensuring all necessary libraries (e.g., `transformers`, `torch`, `spacy`, `pytextrank`, `nltk`) are installed and compatible can be tricky. This often involves `pip install` commands and sometimes specific versions.

3.  **Model Loading and Initialization:** Correctly loading pre-trained models and tokenizers (e.g., `PegasusForConditionalGeneration`, `PegasusTokenizer`) and configuring them for the available device (CPU/GPU) is crucial. Errors can arise from incorrect model names, network issues during download, or device allocation problems.

4.  **Handling `NameError` and Scope:** During iterative development in notebooks, it's common to encounter `NameError` if variables or functions are not defined in the current execution context or if cells are run out of order. Ensuring all necessary imports, DataFrame loads, and function definitions are present and executed before their use is vital.

5.  **Spacy Model Downloads:** SpaCy models (like `en_core_web_lg`) are large and need to be downloaded separately. An `OSError` indicating a missing model requires explicitly running `!python -m spacy download <model_name>`.

6.  **NLTK Resource Downloads:** Similarly, NLTK components (like `stopwords` or `punkt_tab`) need to be downloaded using `nltk.download()` to avoid `LookupError`.

7.  **Hugging Face Authentication (Optional but Recommended):** While not strictly necessary for public models, repeated warnings about missing `HF_TOKEN` suggest that authenticating with the Hugging Face Hub could lead to faster downloads and higher rate limits, improving overall experience, especially for frequent usage.

8.  **Output Interpretation and Customization:** While models provide raw output, converting it into a structured, readable format (like DataFrames or well-formatted print statements) for analysis and comparison requires additional coding. Customizing summarization length (`max_len`) and implementing post-processing steps (like deduplication) are also part of this.

Overall, successfully implementing transformer models involves a combination of environment setup, careful coding, and debugging to navigate library dependencies, resource management, and model-specific configurations.

TEXT CLUSTER

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import pandas as pd
import nltk
from nltk.corpus import stopwords

# Ensure NLTK stopwords are downloaded
nltk.download('stopwords', quiet=True)

# --- 1. Preprocessing and TF-IDF Vectorization ---

# Filter out non-string snippets to avoid errors
corpus_for_clustering = reviews['snippet'].astype(str).tolist()

# Define custom stopwords (can be expanded based on your data)
custom_stopwords = stopwords.words('english') + ['restaurant', 'buffalowildwings', 'bww', 'place', 'like', 'get', 'got', 'go', 'us', 'staff', 'manager', 'just', 'really', 'much', 'would', 'also']

tfidf_vectorizer = TfidfVectorizer(
    max_df=0.85, # ignore terms that appear in more than 85% of documents
    min_df=2,    # ignore terms that appear in less than 2 documents
    stop_words=custom_stopwords
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_for_clustering)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# --- 2. K-Means Clustering ---

# Determine the number of clusters. You might need to experiment with this value.
# A common approach is to use the elbow method or silhouette score.
num_clusters = 4 # Updated based on Elbow Method analysis

km = KMeans(n_clusters=num_clusters, init='k-means++', max_iter=100, n_init=10, random_state=42)
km.fit(tfidf_matrix)

# Assign cluster labels to the DataFrame
reviews['cluster_label'] = km.labels_

print("\nReview snippets clustered successfully!")

In [ ]:
# --- 3. Analyze Clusters ---

# Function to print the top terms for each cluster
def print_top_terms(model, vectorizer, num_words=10):
    feature_names = vectorizer.get_feature_names_out()
    for i, centroid in enumerate(model.cluster_centers_):
        top_words_indices = centroid.argsort()[-num_words:][::-1]
        top_words = [feature_names[j] for j in top_words_indices]
        print(f"Cluster {i+1} Top Terms: {', '.join(top_words)}")

print("\nTop terms for each cluster:")
print_top_terms(km, tfidf_vectorizer)

print("\nDistribution of reviews per cluster:")
display(reviews['cluster_label'].value_counts().sort_index())

## Elbow Method for Optimal K-Means Clusters

The Elbow Method is a heuristic used to determine the optimal number of clusters (`k`) for K-Means clustering. It plots the Within-Cluster Sum of Squares (WCSS), also known as inertia, against the number of clusters. The 'elbow' point on the plot, where the rate of decrease in WCSS significantly changes, is considered an indicator of the optimal `k`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# Assuming tfidf_matrix is already defined from the previous clustering steps.
# If the kernel was restarted or this cell is run independently, you might need to re-run
# the TF-IDF vectorization step (cell with id `3a9b3eb3`) to define `tfidf_matrix`.

wcss = []
# Try different numbers of clusters, typically from 1 to a reasonable maximum (e.g., 10 or 15)
max_clusters = 10

for i in range(1, max_clusters + 1):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(tfidf_matrix)
    wcss.append(kmeans.inertia_)

# Plot the Elbow Method graph
plt.figure(figsize=(10, 6))
sns.lineplot(x=range(1, max_clusters + 1), y=wcss, marker='o', linestyle='--')
plt.title('Elbow Method For Optimal K-Means Clusters', fontsize=15)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12)
plt.xticks(range(1, max_clusters + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print("Look for the 'elbow' point in the graph where the rate of decrease in WCSS slows down significantly. This point often indicates a reasonable number of clusters.")

### Interpreting the Elbow Method Results

The Elbow Method aims to find the 'elbow' point in the plot of Within-Cluster Sum of Squares (WCSS) versus the number of clusters (K). The WCSS measures the sum of squared distances between each point and its assigned cluster centroid. As K increases, the WCSS generally decreases because points are closer to their respective centroids.

In our plot:
*   We see a **sharp decrease** in WCSS as K increases from 1 to 2 or 3.
*   The rate of decrease then **slows down** significantly around K=3, and continues to decrease more gradually thereafter.

This 'elbow' is often visually identifiable as the point where the curve bends. Based on our current plot, an optimal number of clusters for this dataset could reasonably be considered **3 or 4**, as this is where the diminishing returns for adding more clusters become more apparent. While 5 clusters were initially chosen for our K-Means analysis, reviewing this plot suggests that a slightly lower number might also be effective in capturing the main themes within the reviews, without adding unnecessary complexity.

In [ ]:
import pandas as pd
from collections import Counter
import numpy as np
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import spacy

# --- 0. Ensure NLTK and SpaCy resources are ready ---
nltk.download('vader_lexicon', quiet=True)

# --- 1. Load Reviews DataFrame ---
reviews = pd.read_csv('reviews.csv')

# --- 2. Perform Sentiment Analysis ---
sid = SentimentIntensityAnalyzer()
def sentiment(review):
  if(not isinstance(review, str)):
    return [np.nan,np.nan,np.nan,np.nan]
  sentiment_score = sid.polarity_scores(review)
  return [sentiment_score['neg'],sentiment_score['neu'],sentiment_score['pos'],sentiment_score['compound']]
reviews[['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound']] = reviews['snippet'].apply(sentiment).apply(pd.Series)

# --- 3. Extract Entities ---
try:
    nlp
except NameError:
    !python -m spacy download en_core_web_lg --quiet
    nlp = spacy.load("en_core_web_lg")

def get_ents(column):
    return [[(ent.text, ent.label_) for ent in doc.ents]
            for doc in nlp.pipe(column.astype(str))]

reviews['entities'] = get_ents(reviews['snippet'])

# --- 4. Create entities_df ---
all_entities = []
for entity_list in reviews['entities']:
    all_entities.extend(entity_list)
entities_df = pd.DataFrame(all_entities, columns=['value', 'type'])

# --- 5. Correct entity types in entities_df ---
entities_df.loc[entities_df['value'].str.contains('Dakota', case=False), 'type'] = 'GPE'
entities_df.loc[entities_df['value'].str.contains('Maryam', case=False), 'type'] = 'PERSON'

# --- 6. Create entities_with_review_context_df ---
entity_review_data = []
for index, row in reviews.iterrows():
    review_id = row['review_id']
    rating = row['rating']
    sentiment_compound = row['sentiment_compound']
    date = row['date']
    iso_date = row['iso_date']
    snippet = row['snippet']

    entities_in_review = row['entities']

    for entity_value, entity_type in entities_in_review:
        entity_review_data.append({
            'entity_value': entity_value,
            'entity_type': entity_type,
            'review_id': review_id,
            'rating': rating,
            'sentiment_compound': sentiment_compound,
            'date': date,
            'iso_date': iso_date,
            'snippet': snippet
        })
entities_with_review_context_df = pd.DataFrame(entity_review_data)

# --- 7. Correct entity types in entities_with_review_context_df ---
entities_with_review_context_df.loc[entities_with_review_context_df['entity_value'].str.contains('Dakota', case=False), 'entity_type'] = 'GPE'
entities_with_review_context_df.loc[entities_with_review_context_df['entity_value'].str.contains('Maryam', case=False), 'entity_type'] = 'PERSON'

# --- 1. Filter Reviews into Positive and Negative Categories ---
# Define thresholds for positive, neutral, and negative sentiment based on compound score
# These thresholds are common for VADER sentiment analysis, but can be adjusted.
positive_threshold = 0.05
negative_threshold = -0.05

positive_reviews = reviews[reviews['sentiment_compound'] >= positive_threshold]
negative_reviews = reviews[reviews['sentiment_compound'] <= negative_threshold]

print(f"Number of positive reviews: {len(positive_reviews)}")
print(f"Number of negative reviews: {len(negative_reviews)}")

# --- 2. Identify Most Frequent Entities in Positive Reviews ---
def get_all_entities_from_df(df):
    all_entities = []
    for entity_list in df['entities']:
        all_entities.extend(entity_list)
    return pd.DataFrame(all_entities, columns=['value', 'type'])

positive_entities_df = get_all_entities_from_df(positive_reviews)

print("\n--- Most Frequent Entities in Positive Reviews ---")
relevant_entity_types = ['PERSON', 'GPE', 'ORG', 'PRODUCT'] # Focus on these types

for entity_type in relevant_entity_types:
    if not positive_entities_df.empty and entity_type in positive_entities_df['type'].unique():
        filtered_entities = positive_entities_df[positive_entities_df['type'] == entity_type]['value'].str.lower()
        entity_counts = Counter(filtered_entities)
        print(f"\nTop 10 '{entity_type}' (Positive Reviews):")
        for entity, count in entity_counts.most_common(10):
            print(f"- {entity}: {count}")
    else:
        print(f"\nNo '{entity_type}' entities found in positive reviews or DataFrame is empty.")

# --- 3. Identify Most Frequent Entities in Negative Reviews ---
negative_entities_df = get_all_entities_from_df(negative_reviews)

print("\n--- Most Frequent Entities in Negative Reviews ---")
for entity_type in relevant_entity_types:
    if not negative_entities_df.empty and entity_type in negative_entities_df['type'].unique():
        filtered_entities = negative_entities_df[negative_entities_df['type'] == entity_type]['value'].str.lower()
        entity_counts = Counter(filtered_entities)
        print(f"\nTop 10 '{entity_type}' (Negative Reviews):")
        for entity, count in entity_counts.most_common(10):
            print(f"- {entity}: {count}")
    else:
        print(f"\nNo '{entity_type}' entities found in negative reviews or DataFrame is empty.")

# --- 4. Calculate Average Sentiment for Frequently Mentioned Entities ---
print("\n--- Average Sentiment for Frequently Mentioned Entities ---")

# Ensure 'entities_with_review_context_df' is up-to-date and exists
# (This DataFrame should have been created and updated in previous steps)
if 'entities_with_review_context_df' in locals() or 'entities_with_review_context_df' in globals():
    average_sentiment_by_entity = entities_with_review_context_df.groupby('entity_value')['sentiment_compound'].mean().sort_values(ascending=False)

    print("\nTop 10 Entities by Average Compound Sentiment (Most Positive):")
    print(average_sentiment_by_entity.head(10))

    print("\nTop 10 Entities by Average Compound Sentiment (Most Negative):")
    print(average_sentiment_by_entity.tail(10))
else:
    print("Cannot calculate average sentiment: 'entities_with_review_context_df' not found or not correctly populated.")

print("\nAnalysis complete. Review the outputs above to understand entity mentions and their associated sentiment.")